In [ ]:
# Run once per session. Existing verified Python files are reused offline.
# Enable Kaggle Internet only when code needs downloading; no dataset scan here.
import os
import sys
import time
import hashlib
import urllib.request
from pathlib import Path

setup_started = time.monotonic()
def setup_log(message):
    print(f"[{time.monotonic() - setup_started:.1f}s] {message}", flush=True)

setup_log("Starting PCB setup")
WORKING_DIR = Path(os.environ.get("PCB_WORKING_DIR", "/kaggle/working"))
CODE_DIR = WORKING_DIR / "pcb_code"
INPUT_DIR = Path(os.environ.get("PCB_INPUT_DIR", "/kaggle/input/datasets/andreimunteanu"))
DATA_ROOT = Path(os.environ.get("PCB_DATA_ROOT", str(INPUT_DIR / "multi-topology-pcb")))
SMOKE_TEST = os.environ.get("PCB_SMOKE_TEST", "0") == "1"

# No recursive search, split-directory probes, or automatic pip installs.
# If your GitHub import adds a wrapper folder, include it in DATA_ROOT above.
setup_log(f"Dataset path configured: {DATA_ROOT}")
CODE_DIR.mkdir(parents=True, exist_ok=True)
CODE_REVISION = "e7846f3"  # tested model + manifest-based dataset loader
CODE_URL = f"https://raw.githubusercontent.com/ongisho/surrogates/{CODE_REVISION}/pcb"
CODE_SHA256 = {'transolver.py': '7729c3430204a24f8b729ccab208f322d455b6c1b56ac7107b6c19523697528f', 'dataset.py': '056fa9d9f28db5b39e4e91aa250768470f88145a964bcbaf9dc91e40dded1017'}

for name, expected_hash in CODE_SHA256.items():
    destination = CODE_DIR / name
    if destination.is_file() and hashlib.sha256(destination.read_bytes()).hexdigest() == expected_hash:
        setup_log(f"Using cached {name}; no download")
        continue
    setup_log(f"Downloading {name} (10-second socket timeout)...")
    try:
        with urllib.request.urlopen(f"{CODE_URL}/{name}", timeout=10) as response:
            content = response.read()
    except Exception as exc:
        raise RuntimeError(
            f"Could not download {name}. Enable Internet in Kaggle settings, then rerun this cell."
        ) from exc
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError(f"Unexpected contents for {name}; cached code was left untouched.")
    compile(content, str(destination), "exec")
    temporary = destination.with_suffix(".py.tmp")
    temporary.write_bytes(content)
    temporary.replace(destination)
    setup_log(f"Saved {name}")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
setup_log("Setup complete; dataset files have not been scanned")


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from pathlib import Path
import json
import time
from dataset import PCBDataset, NormalizationStats, discover_files
from transolver import Transolver, PhysicsAttentionIrregularMesh, TransolverBlock

## 1. Hyperparameters

In [ ]:
HPARAMS = {
    # data: official train/validation/test directories
    "data_root": str(DATA_ROOT),
    "train_fraction": 0.2,  # reproducible 20% subset; validation/test stay complete
    "stride": 1,  # native 256 x 256 (higher GPU memory)

    # model: standard Transolver, no recursive state
    "space_dim": 2,
    "fun_dim": 6,
    "out_dim": 1,
    "n_hidden": 128,
    "n_layers": 8,
    "n_heads": 8,
    "slice_num": 64,
    "mlp_ratio": 1,
    "dropout": 0.0,
    "act": "gelu",

    # training
    "batch_size": 16,
    "epochs": 200,
    "lr": 1e-3,
    "weight_decay": 0,  # 1e-5,
    "max_grad_norm": 0.1,

    # runtime
    "num_workers": 0,
    "pin_memory": True,
    "seed": 42,

    # output
    "checkpoint_path": str(WORKING_DIR / "pcb_transolver.pt"),
    "last_checkpoint_path": str(WORKING_DIR / "pcb_transolver_last.pt"),
    "debug_csv_path": str(WORKING_DIR / "pcb_transolver_debug_log.csv"),
}

DEBUG = {
    "stop_on_nonfinite": True,
    "log_every_n_batches": 20,
    "print_every_n_batches": 20,
    "max_train_batches": None,
    "max_eval_batches": None,
    "detect_anomaly": False,
    "save_parameter_snapshots": True,
}

if SMOKE_TEST:
    HPARAMS.update(n_hidden=32, n_layers=2, n_heads=4, slice_num=8,
                   epochs=1, batch_size=1)
    torch.set_num_threads(2)

torch.autograd.set_detect_anomaly(DEBUG["detect_anomaly"])

## 2. Diagnostics

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def tensor_stats(x):
    x = x.detach()
    finite = torch.isfinite(x)

    out = {
        "finite_fraction": finite.float().mean().item(),
        "nan_count": torch.isnan(x).sum().item(),
        "inf_count": torch.isinf(x).sum().item(),
    }

    if finite.any():
        xf = x[finite].float()
        out.update({
            "mean": xf.mean().item(),
            "std": xf.std(unbiased=False).item(),
            "rms": torch.sqrt(torch.mean(xf * xf)).item(),
            "absmax": xf.abs().max().item(),
            "norm": xf.norm().item(),
            "min": xf.min().item(),
            "max": xf.max().item(),
        })
    else:
        out.update({
            "mean": np.nan, "std": np.nan, "rms": np.nan,
            "absmax": np.nan, "norm": np.nan,
            "min": np.nan, "max": np.nan,
        })

    return out


def check_finite(name, x, context=None):
    if torch.isfinite(x).all():
        return

    context = context or {}
    print("\n" + "=" * 90)
    print("NON-FINITE TENSOR:", name)
    print("context:", context)
    print("shape:", tuple(x.shape))
    print("stats:", tensor_stats(x))
    print("=" * 90)

    if DEBUG["stop_on_nonfinite"]:
        raise FloatingPointError(f"{name} contains NaN/Inf at {context}")


def flatten_finite_tensors(tensors):
    vals = []
    for t in tensors:
        if t is None:
            continue
        a = t.detach().float().reshape(-1).cpu()
        a = a[torch.isfinite(a)]
        if a.numel():
            vals.append(a)
    return torch.cat(vals) if vals else torch.empty(0)


def parameter_group_map(model):
    return {
        "input_encoder": list(model.preprocess.parameters()) + [model.placeholder],
        "blocks": [p for block in model.blocks for name, p in block.named_parameters()
                   if not name.startswith(("ln_3.", "output."))],
        "decoder": list(model.blocks[-1].ln_3.parameters()) + list(model.blocks[-1].output.parameters()),
    }


def stats_from_flat(flat):
    if flat.numel() == 0:
        return {
            "mean": np.nan, "std": np.nan, "rms": np.nan,
            "absmax": np.nan, "norm": np.nan,
        }
    return {
        "mean": flat.mean().item(),
        "std": flat.std(unbiased=False).item(),
        "rms": torch.sqrt(torch.mean(flat * flat)).item(),
        "absmax": flat.abs().max().item(),
        "norm": flat.norm().item(),
    }


def group_parameter_stats(params):
    return stats_from_flat(
        flatten_finite_tensors([p.data for p in params])
    )


def group_gradient_stats(params):
    return stats_from_flat(
        flatten_finite_tensors([p.grad for p in params if p.grad is not None])
    )


def snapshot_parameter_values(model):
    return {
        name: flatten_finite_tensors([p.data for p in params]).numpy()
        for name, params in parameter_group_map(model).items()
    }


def snapshot_gradient_values(model):
    return {
        name: flatten_finite_tensors(
            [p.grad for p in params if p.grad is not None]
        ).numpy()
        for name, params in parameter_group_map(model).items()
    }


set_seed(HPARAMS["seed"])

## 3. PCB dataset

In [ ]:
# Resolve an optional immediate GitHub repository wrapper only when data are needed.
if not (DATA_ROOT / "TrainData").is_dir() and DATA_ROOT.is_dir():
    wrappers = [p for p in DATA_ROOT.iterdir() if p.is_dir()
                and all((p / split).is_dir() for split in ("TrainData", "ValData", "TestData"))]
    if len(wrappers) == 1:
        DATA_ROOT = wrappers[0]
        HPARAMS["data_root"] = str(DATA_ROOT)

print("Discovering training files...", flush=True)
all_train_files = discover_files(HPARAMS["data_root"], "TrainData")
train_fraction = HPARAMS["train_fraction"]
if not 0 < train_fraction <= 1:
    raise ValueError("train_fraction must be in (0, 1]")
n_train = max(1, int(len(all_train_files) * train_fraction))
indices = np.sort(np.random.default_rng(HPARAMS["seed"]).choice(
    len(all_train_files), size=n_train, replace=False,
))
train_files = [all_train_files[i] for i in indices]
print(f"Using {len(train_files)}/{len(all_train_files)} training files "
      f"({100 * train_fraction:g}%, seed={HPARAMS['seed']})", flush=True)
print("Discovering validation files...", flush=True)
val_files = discover_files(HPARAMS["data_root"], "ValData")
print("Discovering test files...", flush=True)
test_files = discover_files(HPARAMS["data_root"], "TestData")

# Compute exact statistics from ALL selected training files, then reuse the cache.
# Validation/test targets and unselected training files are not used for fitting.
NORM_MAX_FILES = None
normalizer = NormalizationStats.fit(
    train_files, max_files=NORM_MAX_FILES, seed=HPARAMS["seed"],
    cache_path=WORKING_DIR / "pcb_normalization.json", progress_every=25,
)
train_dataset = PCBDataset(HPARAMS["data_root"], "TrainData", normalizer, HPARAMS["stride"], train_files)
val_dataset = PCBDataset(HPARAMS["data_root"], "ValData", normalizer, HPARAMS["stride"], val_files)
test_dataset = PCBDataset(HPARAMS["data_root"], "TestData", normalizer, HPARAMS["stride"], test_files)

def make_loader(dataset, shuffle=False):
    return DataLoader(dataset, batch_size=HPARAMS["batch_size"], shuffle=shuffle,
                      num_workers=HPARAMS["num_workers"], pin_memory=HPARAMS["pin_memory"])

train_loader = make_loader(train_dataset, shuffle=True)
val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Coordinates / features / target:", [tuple(a.shape) for a in train_dataset[0]])
print(normalizer)

## 4. Debug recorder

In [ ]:
class DebugRecorder:
    def __init__(self):
        self.rows = []

    def record_tensor(self, tag, x, context):
        row = dict(context)
        row.update({"kind": "tensor", "tag": tag, **tensor_stats(x)})
        self.rows.append(row)

    def record_scalar(self, tag, value, context):
        row = dict(context)
        row.update({"kind": "scalar", "tag": tag, "value": float(value)})
        self.rows.append(row)

    def dataframe(self):
        return pd.DataFrame(self.rows)


recorder = DebugRecorder()

## 5. Transolver and diagnostics hooks

In [ ]:
# Network implementation is downloaded from pcb/transolver.py in cell 0.
# Hooks inspect standard Transolver without replacing its architecture.
diagnostic_context = None
hook_handles = []

def block_hook(name):
    def hook(module, inputs, output):
        if diagnostic_context is not None:
            check_finite(name, output, diagnostic_context)
            recorder.record_tensor(name, output, diagnostic_context)
    return hook

def slice_hook(attention, name):
    def hook(module, inputs, output):
        if diagnostic_context is None:
            return
        with torch.no_grad():
            logits = output.detach().float() / attention.temperature.detach().float()
            weights = logits.softmax(dim=-1)
            norm = weights.sum(dim=2)
            entropy = -(weights.clamp_min(1e-12) * weights.clamp_min(1e-12).log()).sum(-1).mean()
            check_finite(name + ".slice_logits", logits, diagnostic_context)
            values = {
                "temperature_min": attention.temperature.detach().min().item(),
                "temperature_max": attention.temperature.detach().max().item(),
                "temperature_absmin": attention.temperature.detach().abs().min().item(),
                "slice_logits_absmax": logits.abs().max().item(),
                "slice_norm_min": norm.min().item(),
                "slice_norm_max": norm.max().item(),
                "slice_entropy": entropy.item(),
            }
            for tag, value in values.items():
                recorder.record_scalar(tag, value, dict(diagnostic_context, layer=name))
    return hook

## 6. Instantiate model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_kwargs = {key: HPARAMS[key] for key in (
    "space_dim", "fun_dim", "out_dim", "n_hidden", "n_layers", "n_heads",
    "slice_num", "mlp_ratio", "dropout", "act")}
model = Transolver(**model_kwargs).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=HPARAMS["lr"],
                              weight_decay=HPARAMS["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=HPARAMS["epochs"])

for name, module in model.named_modules():
    if isinstance(module, TransolverBlock):
        hook_handles.append(module.register_forward_hook(block_hook(name)))
    if isinstance(module, PhysicsAttentionIrregularMesh):
        hook_handles.append(module.in_project_slice.register_forward_hook(slice_hook(module, name)))

initial_parameter_snapshot = snapshot_parameter_values(model) if DEBUG["save_parameter_snapshots"] else {}
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print(model)

## 7. Loss and metric

In [ ]:
def relative_l2_per_sample(pred, target):
    B = pred.shape[0]

    pred = pred.reshape(B, -1)
    target = target.reshape(B, -1)

    diff_norm = torch.norm(
        pred - target,
        p=2,
        dim=1,
    )

    target_norm = torch.norm(
        target,
        p=2,
        dim=1,
    )

    return diff_norm / target_norm


def relative_l2_loss(pred, target):
    return relative_l2_per_sample(
        pred,
        target,
    ).sum()


def mae_metric(pred, target):
    return torch.mean(
        torch.abs(pred - target)
    )

## 8. Dry run — inspect layer magnitudes before training

In [ ]:
model.eval()
coords, features, target_norm = [a.to(device) for a in next(iter(train_loader))]
diagnostic_context = {"epoch": 0, "batch": 0, "phase": "dry_run"}
with torch.no_grad():
    pred_norm = model(coords, features)
    pred = normalizer.decode_temperature(pred_norm)
    target = normalizer.decode_temperature(target_norm)
    check_finite("dry_run.prediction", pred, diagnostic_context)
    print("Prediction:", tuple(pred.shape), tensor_stats(pred))
    print("Relative L2:", relative_l2_loss(pred, target).item())
    print("MAE (C):", mae_metric(pred, target).item())
diagnostic_context = None
state_rows = recorder.dataframe()
state_rows = state_rows[state_rows["kind"] == "tensor"]
if len(state_rows):
    plt.figure(figsize=(9, 5))
    plt.plot(np.arange(len(state_rows)), state_rows["rms"], marker="o")
    plt.xlabel("Transolver block"); plt.ylabel("RMS")
    plt.title("Dry run: layer output RMS"); plt.grid(True); plt.tight_layout(); plt.show()

## 9. Training and evaluation with diagnostics

In [ ]:
history = {"train_rel_l2": [], "val_rel_l2": [], "train_mae": [], "val_mae": [], "lr": []}
step_log = []
last_gradient_snapshot = None

def current_temperature_stats(model):
    temps = torch.cat([m.temperature.detach().flatten() for m in model.modules()
                       if isinstance(m, PhysicsAttentionIrregularMesh)])
    return temps.min().item(), temps.max().item(), temps.abs().min().item()

def add_group_stats(model, row):
    for group_name, params in parameter_group_map(model).items():
        for prefix, values in (("param", group_parameter_stats(params)),
                               ("grad", group_gradient_stats(params))):
            for key, value in values.items():
                row[f"{prefix}_{group_name}_{key}"] = value

def train_one_epoch_debug(epoch):
    global last_gradient_snapshot, diagnostic_context
    model.train()
    total_rel_l2 = total_mae = total_samples = 0
    for batch_idx, batch in enumerate(train_loader):
        if DEBUG["max_train_batches"] is not None and batch_idx >= DEBUG["max_train_batches"]:
            break
        coords, features, target_norm = [a.to(device, non_blocking=True) for a in batch]
        target = normalizer.decode_temperature(target_norm)
        context = {"epoch": epoch, "batch": batch_idx, "phase": "train"}
        detailed = batch_idx % DEBUG["log_every_n_batches"] == 0
        diagnostic_context = context if detailed else None
        optimizer.zero_grad(set_to_none=True)
        pred_norm = model(coords, features)
        diagnostic_context = None
        pred = normalizer.decode_temperature(pred_norm)
        loss = relative_l2_loss(pred, target)
        check_finite("train.loss", loss.reshape(1), context)
        loss.backward()
        raw_grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), HPARAMS["max_grad_norm"],
                                                       error_if_nonfinite=True)
        row = dict(context, global_optimizer_step=len(step_log), loss_rel_l2=loss.item(),
                   mae=mae_metric(pred, target).item(), raw_grad_norm=raw_grad_norm.item(),
                   pred_absmax=pred.detach().abs().max().item())
        row["temperature_min"], row["temperature_max"], row["temperature_absmin"] = current_temperature_stats(model)
        if detailed:
            add_group_stats(model, row)
            if DEBUG["save_parameter_snapshots"]:
                last_gradient_snapshot = snapshot_gradient_values(model)
        step_log.append(row)
        optimizer.step()
        for name, parameter in model.named_parameters():
            check_finite("parameter." + name, parameter, context)
        batch_size = coords.shape[0]
        total_rel_l2 += relative_l2_per_sample(pred.detach(), target).sum().item()
        total_mae += row["mae"] * batch_size
        total_samples += batch_size
        if batch_idx % DEBUG["print_every_n_batches"] == 0:
            print(f"epoch={epoch:03d} batch={batch_idx:04d} loss={row['loss_rel_l2']:.6g} "
                  f"grad={row['raw_grad_norm']:.3g} Tmin={row['temperature_min']:.3g}")
    if not total_samples:
        raise ValueError("No training batches selected")
    return {"rel_l2": total_rel_l2 / total_samples, "mae": total_mae / total_samples}

@torch.no_grad()
def evaluate_debug(loader):
    model.eval()
    total_rel_l2 = total_mae = total_samples = 0
    for batch_idx, batch in enumerate(loader):
        if DEBUG["max_eval_batches"] is not None and batch_idx >= DEBUG["max_eval_batches"]:
            break
        coords, features, target_norm = [a.to(device, non_blocking=True) for a in batch]
        target = normalizer.decode_temperature(target_norm)
        pred = normalizer.decode_temperature(model(coords, features))
        check_finite("evaluation.prediction", pred, {"batch": batch_idx})
        batch_size = coords.shape[0]
        total_rel_l2 += relative_l2_per_sample(pred, target).sum().item()
        total_mae += mae_metric(pred, target).item() * batch_size
        total_samples += batch_size
    if not total_samples:
        raise ValueError("No evaluation batches selected")
    return {"rel_l2": total_rel_l2 / total_samples, "mae": total_mae / total_samples}

## 10. Run training

In [ ]:
training_failed = False
failure_message = None
best_val = float("inf")
try:
    for epoch in range(1, HPARAMS["epochs"] + 1):
        train_metrics = train_one_epoch_debug(epoch)
        val_metrics = evaluate_debug(val_loader)
        scheduler.step()
        for split, metrics in (("train", train_metrics), ("val", val_metrics)):
            history[split + "_rel_l2"].append(metrics["rel_l2"])
            history[split + "_mae"].append(metrics["mae"])
        history["lr"].append(optimizer.param_groups[0]["lr"])
        checkpoint = {"model_state_dict": model.state_dict(), "model_kwargs": model_kwargs,
                      "optimizer_state_dict": optimizer.state_dict(),
                      "scheduler_state_dict": scheduler.state_dict(), "epoch": epoch,
                      "normalization": normalizer.to_dict(), "hparams": HPARAMS,
                      "history": history, "val_metrics": val_metrics}
        torch.save(checkpoint, HPARAMS["last_checkpoint_path"])
        if val_metrics["rel_l2"] < best_val:
            best_val = val_metrics["rel_l2"]
            torch.save(checkpoint, HPARAMS["checkpoint_path"])
        print(f"\nEpoch {epoch:03d}/{HPARAMS['epochs']} | "
              f"Train RelL2: {train_metrics['rel_l2']:.6f} | Val RelL2: {val_metrics['rel_l2']:.6f} | "
              f"Train MAE: {train_metrics['mae']:.6f} | Val MAE: {val_metrics['mae']:.6f} | "
              f"LR: {optimizer.param_groups[0]['lr']:.3e}\n")
except (FloatingPointError, RuntimeError) as exc:
    training_failed = True
    failure_message = str(exc)
    print("TRAINING STOPPED:", failure_message)
    raise
finally:
    diagnostic_context = None
    step_df = pd.DataFrame(step_log)
    step_df.to_csv(HPARAMS["debug_csv_path"], index=False)
    print("Debug CSV:", HPARAMS["debug_csv_path"])
print("training_failed:", training_failed)

## 11. Core plots

In [ ]:
def plot_series(df, columns, ylabel, title, logy=False):
    if df is None or len(df) == 0:
        print("No data for:", title)
        return

    plt.figure(figsize=(9, 5))

    for col in columns:
        if col in df.columns:
            plt.plot(
                df["global_optimizer_step"],
                df[col],
                label=col,
            )

    plt.xlabel("Optimizer step")
    plt.ylabel(ylabel)
    plt.title(title)

    if logy:
        plt.yscale("log")

    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


if len(step_log):
    step_df = pd.DataFrame(step_log)

    plot_series(
        step_df,
        ["loss_rel_l2"],
        "Relative L2",
        "Training loss",
        logy=True,
    )

    plot_series(
        step_df,
        ["raw_grad_norm"],
        "Gradient norm before clipping",
        "Raw gradient norm",
        logy=True,
    )

    plot_series(
        step_df,
        ["temperature_min", "temperature_max", "temperature_absmin"],
        "Temperature",
        "Physics-Attention temperature",
    )

## 12. Gradients and parameter values

In [ ]:
if len(step_log):
    # Gradient norms
    plot_series(
        step_df,
        [
            "grad_input_encoder_norm",
            "grad_blocks_norm",
            "grad_decoder_norm",
        ],
        "Gradient L2 norm",
        "Gradient norm by network section",
        logy=True,
    )

    # Gradient maximums
    plot_series(
        step_df,
        [
            "grad_input_encoder_absmax",
            "grad_blocks_absmax",
            "grad_decoder_absmax",
        ],
        "Max |gradient|",
        "Maximum absolute gradient by network section",
        logy=True,
    )

    # Parameter norms
    plot_series(
        step_df,
        [
            "param_input_encoder_norm",
            "param_blocks_norm",
            "param_decoder_norm",
        ],
        "Parameter L2 norm",
        "Parameter norm by network section",
        logy=True,
    )

    # Parameter maximums
    plot_series(
        step_df,
        [
            "param_input_encoder_absmax",
            "param_blocks_absmax",
            "param_decoder_absmax",
        ],
        "Max |parameter|",
        "Maximum absolute parameter by network section",
        logy=True,
    )

In [ ]:
final_parameter_snapshot = snapshot_parameter_values(model)

for group_name, final_values in final_parameter_snapshot.items():
    initial_values = initial_parameter_snapshot.get(
        group_name,
        np.array([]),
    )

    plt.figure(figsize=(8, 4))

    if len(initial_values):
        plt.hist(
            initial_values,
            bins=80,
            alpha=0.5,
            label="initial",
        )

    if len(final_values):
        plt.hist(
            final_values,
            bins=80,
            alpha=0.5,
            label="current/final",
        )

    plt.xlabel("Parameter value")
    plt.ylabel("Count")
    plt.title(f"Parameter distribution — {group_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


if last_gradient_snapshot is not None:
    for group_name, values in last_gradient_snapshot.items():
        if len(values) == 0:
            continue

        plt.figure(figsize=(8, 4))
        plt.hist(values, bins=100)
        plt.xlabel("Gradient value")
        plt.ylabel("Count")
        plt.title(f"Latest gradient distribution — {group_name}")
        plt.grid(True)
        plt.tight_layout()
        plt.show()

## 13. Physics-Attention internals

In [ ]:
internal_df = recorder.dataframe()

if len(internal_df):
    scalar_df = internal_df[
        internal_df["kind"] == "scalar"
    ].copy()

    def plot_internal(tag, logy=False):
        d = scalar_df[
            scalar_df["tag"] == tag
        ].reset_index(drop=True)

        if len(d) == 0:
            print("No records for", tag)
            return

        plt.figure(figsize=(9, 4))
        plt.plot(np.arange(len(d)), d["value"])
        plt.xlabel("Logged Physics-Attention call")
        plt.ylabel(tag)
        plt.title(tag)

        if logy:
            plt.yscale("log")

        plt.grid(True)
        plt.tight_layout()
        plt.show()

    plot_internal("temperature_min")
    plot_internal("temperature_absmin")
    plot_internal("slice_logits_absmax", logy=True)
    plot_internal("slice_norm_min", logy=True)
    plot_internal("slice_norm_max", logy=True)
    plot_internal("slice_entropy")
else:
    print("No detailed internal records.")

## 14. Layer output evolution

In [ ]:
internal_df = recorder.dataframe()
tensor_df = internal_df[internal_df["kind"] == "tensor"].copy()
if len(tensor_df):
    latest_epoch = tensor_df["epoch"].max()
    d = tensor_df[tensor_df["epoch"] == latest_epoch]
    d = d[d["batch"] == d["batch"].max()]
    display(d[["epoch", "batch", "tag", "mean", "std", "rms", "absmax", "norm"]])
    for metric in ("rms", "absmax"):
        plt.figure(figsize=(9, 5))
        plt.plot(np.arange(len(d)), d[metric], marker="o")
        plt.xlabel("Transolver block"); plt.ylabel(metric)
        plt.title("Layer output " + metric); plt.grid(True); plt.tight_layout(); plt.show()

## 15. Epoch curves

In [ ]:
if len(history["train_rel_l2"]):
    e = np.arange(1, len(history["train_rel_l2"]) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(e, history["train_rel_l2"], label="Train")
    plt.plot(e, history["val_rel_l2"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Relative L2")
    plt.yscale("log")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(e, history["train_mae"], label="Train")
    plt.plot(e, history["val_mae"], label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print("No complete epoch recorded.")

## 16. Interpretation

The model is standard Transolver with pre-LayerNorm residual blocks. The diagnostics track finite activations, gradients, learned slice temperatures, routing entropy, and layer magnitudes. There are no recursive H/L states in this network.

Coordinates are normalized to [0, 1]. Features are training-standardized power, chip occupancy, ceramic/baseplate/copper masks and h/10000. Temperature is standardized using training statistics, then decoded to Celsius for loss and metrics. No target-derived `max_temps` enter the inputs.

`stride=4` uses 64×64 points and may miss fine boundaries. Set `stride=1` for 256×256. Metrics describe the selected resolution. Official splits are retained; layouts may overlap across them, so these results do not demonstrate unseen-layout generalization. Sample-only runs validate execution, not accuracy.

Dataset: [Multi-topology Dataset](https://github.com/henkuailederen/Multi-topology-Dataset), Ruiting Ke, Jianfeng Tao and Chengliang Liu, CC BY 4.0. Cite the repository and accompanying study when using these data.

## 17. Best checkpoint and held-out test predictions

In [ ]:
checkpoint = torch.load(HPARAMS["checkpoint_path"], map_location=device, weights_only=True)
model.load_state_dict(checkpoint["model_state_dict"])
normalizer = NormalizationStats(**checkpoint["normalization"])
test_metrics = evaluate_debug(test_loader)
print("Best epoch:", checkpoint["epoch"])
print("Test Relative L2:", test_metrics["rel_l2"])
print("Test MAE (C):", test_metrics["mae"])
(WORKING_DIR / "pcb_test_metrics.json").write_text(json.dumps(test_metrics, indent=2))

model.eval()
coords, features, target_norm = test_dataset[0]
with torch.no_grad():
    pred_norm = model(coords[None].to(device), features[None].to(device))[0].cpu()
height = len(torch.unique(coords[:, 1]))
true_c = normalizer.decode_temperature(target_norm).numpy().reshape(height, -1)
pred_c = normalizer.decode_temperature(pred_norm).numpy().reshape(true_c.shape)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
lo, hi = min(true_c.min(), pred_c.min()), max(true_c.max(), pred_c.max())
for ax, values, title in zip(axes, [true_c, pred_c, np.abs(pred_c - true_c)],
                            ["Reference (C)", "Prediction (C)", "Absolute error (C)"]):
    im = ax.imshow(values, **({} if "error" in title else dict(vmin=lo, vmax=hi)))
    ax.set_title(title); ax.axis("off"); fig.colorbar(im, ax=ax, shrink=0.75)
plt.tight_layout(); plt.savefig(WORKING_DIR / "pcb_test_prediction.png", dpi=140); plt.show()
np.savez_compressed(WORKING_DIR / "pcb_test_prediction.npz", reference_c=true_c, prediction_c=pred_c)
print("Best checkpoint:", HPARAMS["checkpoint_path"])
for handle in hook_handles:
    handle.remove()